In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

In [0]:
# 1. Load data from Gold tables (Star Schema)
fact_sales = spark.table("workspace.gold.fact_sales")
dim_customer = spark.table("workspace.gold.dim_customers")

In [0]:
# 2. Perform Feature Engineering (Calculate RFM Metrics per Customer)
# Recency: Days since last purchase | Frequency: Total Orders | Monetary: Total Spend
customer_features = fact_sales.groupBy("customer_key").agg(
    F.datediff(F.current_date(), F.max("order_date")).alias("recency"),
    F.countDistinct("order_number").alias("frequency"),
    F.sum("sales_amount").alias("monetary"),
    F.avg("sales_amount").alias("avg_order_value")
)

In [0]:
# 3. Join with Customer Demographics
features_df = customer_features.join(
    dim_customer.select("customer_key", "country", "marital_status", "gender"),
    on="customer_key",
    how="inner"
)

In [0]:
# 4. Save Feature Table to Unity Catalog
features_df.write.format("delta").mode("overwrite").saveAsTable("gold.ml_customer_features")

print("Feature engineering completed. Saved to `gold.ml_customer_features`.")